# SN1987A next steps after supervisor meeting


- `SN1987A_gaia_candidates.ipynb`: generates `output_files/SN1987A_cands.csv`
- `SN1987A_comoving_search.ipynb`: first local co-moving selection
- `SN1987A_R136_geometry.ipynb`: R136/SN1987A geometry sanity check
- `SN1987A_figures.ipynb`: currently empty; can receive the plotting cells below

Do **not** use `SN1987A_search.ipynb` as-is for SN1987A. It is still an R136-runaway pipeline.

## 0. Imports and constants

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.coordinates import SkyCoord, SkyOffsetFrame

# Only needed for fresh Gaia queries.
# Keep this import here so the rest of the notebook still works if you skip the query section.
try:
    from astroquery.gaia import Gaia
    HAS_GAIA = True
except Exception as e:
    HAS_GAIA = False
    print("astroquery.gaia not available:", e)

input_dir = Path("input_files")
output_dir = Path("output_files")
input_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

D_LMC_KPC = 49.59

# Tegkelidis et al. 2025 final SN1987A position, ICRS J2016
sn87a = SkyCoord("05h35m27.9884s", "-69d16m11.1134s", frame="icrs")

RA_87A = sn87a.ra.deg
DEC_87A = sn87a.dec.deg

# Tegkelidis et al. proper motion, mas/yr
PMRA_87A = 1.60
PMDEC_87A = 0.44

# R136 position used in your R136 workflow
r136 = SkyCoord(ra=84.6767920*u.deg, dec=-69.1006110*u.deg, frame="icrs")
PMRA_R136 = 1.654
PMDEC_R136 = 0.573

print("SN1987A:", RA_87A, DEC_87A)
print("R136:", r136.ra.deg, r136.dec.deg)

astroquery.gaia not available: No module named 'astroquery'
SN1987A: 83.86661833333332 -69.26975372222222
R136: 84.676792 -69.100611


## 1. Sanity-check your current `SN1987A_cands.csv`

This is important because `SN1987A_gaia_candidates.ipynb` currently contains old R136-debug cells.
If those are run after loading `SN1987A_raw.csv`, they can overwrite `runaways_raw` with `R136_runaways_raw.csv`.

This cell catches that by checking whether the saved candidates are actually close to SN1987A.

## 2. Drop-in replacement for the risky part of `SN1987A_gaia_candidates.ipynb`

Use this to replace the cells from “Download Gaia data” through “load raw data”.
This prevents accidentally loading `R136_runaways_raw.csv`.

In [ ]:
def query_sn1987a_raw(radius_deg=0.10, g_limit=19.0, overwrite=False):
    if not HAS_GAIA:
        raise ImportError("astroquery.gaia is not available in this environment.")

    raw_file = input_dir / f"SN1987A_raw_r{str(radius_deg).replace('.', 'p')}_g{str(g_limit).replace('.', 'p')}.csv"

    if raw_file.exists() and not overwrite:
        print("Using existing file:", raw_file)
        return pd.read_csv(raw_file, low_memory=False)

    query = f'''
    SELECT
        *,
        tmass.j_m,
        tmass.j_msigcom,
        tmass.h_m,
        tmass.h_msigcom,
        tmass.ks_m,
        tmass.ks_msigcom
    FROM gaiadr3.gaia_source AS dr3
    LEFT JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch
        USING (source_id)
    LEFT JOIN gaiadr3.tmass_psc_xsc_join AS xjoin
        USING (clean_tmass_psc_xsc_oid)
    LEFT JOIN gaiadr1.tmass_original_valid AS tmass
        ON xjoin.original_psc_source_id = tmass.designation
    WHERE 1 = CONTAINS(
        POINT('ICRS', dr3.ra, dr3.dec),
        CIRCLE('ICRS', {RA_87A}, {DEC_87A}, {radius_deg})
    )
    AND dr3.phot_g_mean_mag < {g_limit}
    '''

    print("Running Gaia query...")
    job = Gaia.launch_job_async(
        query,
        dump_to_file=True,
        output_format="csv",
        output_file=str(raw_file)
    )

    df_raw = pd.read_csv(raw_file, low_memory=False)
    print("Saved:", raw_file)
    print("Rows:", len(df_raw))
    return df_raw


# Example:
# runaways_raw = query_sn1987a_raw(radius_deg=0.10, g_limit=19.0, overwrite=False)
# runaways_all = runaways_raw.copy()

## 3. Improved local co-moving search

Your current `SN1987A_comoving_search.ipynb` already finds 135 stars using:
- projected separation < 100 pc
- velocity difference from SN1987A < 30 km/s

This version adds the local median LMC motion, so you can see whether SN1987A is unusual relative to its local Gaia field.

In [ ]:
df = pd.read_csv(output_dir / "SN1987A_cands_checked.csv", low_memory=False)
df.columns = df.columns.str.strip()

coords = SkyCoord(df["ra"].values*u.deg, df["dec"].values*u.deg, frame="icrs")
df["sep_87a_pc"] = sn87a.separation(coords).radian * D_LMC_KPC * 1000.0

def masyr_to_kms(mu_masyr, distance_kpc=D_LMC_KPC):
    return 4.74047 * distance_kpc * mu_masyr

# Local reference sample. Use the inner 100 pc unless it is too sparse.
local_ref = df[df["sep_87a_pc"] < 100].copy()
if len(local_ref) < 20:
    local_ref = df.copy()
    print("Warning: local reference sample small; using full candidate table.")

pmra_local = np.nanmedian(local_ref["pmra"])
pmdec_local = np.nanmedian(local_ref["pmdec"])

pmra_rel_87a = PMRA_87A - pmra_local
pmdec_rel_87a = PMDEC_87A - pmdec_local
vrel_87a_kms = masyr_to_kms(np.hypot(pmra_rel_87a, pmdec_rel_87a))

df["dpm_vs_sn87a_masyr"] = np.hypot(df["pmra"] - PMRA_87A, df["pmdec"] - PMDEC_87A)
df["dv_vs_sn87a_kms"] = masyr_to_kms(df["dpm_vs_sn87a_masyr"])

df["dpm_vs_local_masyr"] = np.hypot(df["pmra"] - pmra_local, df["pmdec"] - pmdec_local)
df["dv_vs_local_kms"] = masyr_to_kms(df["dpm_vs_local_masyr"])

print("Local median PMRA*:", pmra_local)
print("Local median PMDec:", pmdec_local)
print("SN1987A residual PMRA*:", pmra_rel_87a)
print("SN1987A residual PMDec:", pmdec_rel_87a)
print("SN1987A residual speed [km/s]:", vrel_87a_kms)

local_comoving = df[
    (df["sep_87a_pc"] < 100) &
    (df["dv_vs_sn87a_kms"] < 30)
].copy()

local_comoving = local_comoving.sort_values(["dv_vs_sn87a_kms", "sep_87a_pc"])

cols = [
    "source_id", "ra", "dec",
    "phot_g_mean_mag", "bp_rp",
    "pmra", "pmdec",
    "sep_87a_pc",
    "dv_vs_sn87a_kms",
    "dv_vs_local_kms",
    "ruwe"
]
cols = [c for c in cols if c in local_comoving.columns]

print("Local co-moving candidates:", len(local_comoving))
display(local_comoving[cols].head(30))

local_comoving.to_csv(output_dir / "SN1987A_local_comoving_candidates.csv", index=False)
df.to_csv(output_dir / "SN1987A_cands_with_local_motion.csv", index=False)

## 4. Back-project possible SN1987A origin points

This uses the SN1987A residual proper motion relative to the local field.
That is the better first-order approximation than using the absolute proper motion.

In [ ]:
def offset_coord(center, dx_mas, dy_mas):
    frame = SkyOffsetFrame(origin=center)
    shifted = SkyCoord(lon=dx_mas*u.mas, lat=dy_mas*u.mas, frame=frame)
    return shifted.transform_to("icrs")

def backproject_coord(current_coord, pmra_rel, pmdec_rel, age_myr):
    age_yr = age_myr * 1e6
    dx_mas = -pmra_rel * age_yr
    dy_mas = -pmdec_rel * age_yr
    return offset_coord(current_coord, dx_mas, dy_mas)

# 10-14 Myr is motivated by the local 87A population.
# 30-50 Myr tests a lower-mass ~8 Msun progenitor scenario.
ages_myr = np.array([5, 8, 10, 12, 14, 20, 30, 40, 50], dtype=float)

rows = []
for age in ages_myr:
    c = backproject_coord(sn87a, pmra_rel_87a, pmdec_rel_87a, age)
    rows.append({
        "name": f"SN1987A_backprojected_{age:g}Myr",
        "age_myr": age,
        "ra": c.ra.deg,
        "dec": c.dec.deg,
        "pmra_rel_used_masyr": pmra_rel_87a,
        "pmdec_rel_used_masyr": pmdec_rel_87a,
        "vrel_used_kms": vrel_87a_kms
    })

origin_points = pd.DataFrame(rows)

# Add current 87A and R136 as comparison regions.
comparison = pd.DataFrame([
    {"name": "SN1987A_current", "age_myr": 0.0, "ra": RA_87A, "dec": DEC_87A,
     "pmra_rel_used_masyr": np.nan, "pmdec_rel_used_masyr": np.nan, "vrel_used_kms": np.nan},
    {"name": "R136", "age_myr": np.nan, "ra": r136.ra.deg, "dec": r136.dec.deg,
     "pmra_rel_used_masyr": np.nan, "pmdec_rel_used_masyr": np.nan, "vrel_used_kms": np.nan},
])

region_centres = pd.concat([comparison, origin_points], ignore_index=True)
region_centres.to_csv(output_dir / "SN1987A_region_centres_to_query.csv", index=False)

display(region_centres)

## 5. Plot current 87A, R136, and back-projected origin points

These cells can go into `SN1987A_figures.ipynb`.

In [ ]:
region_centres = pd.read_csv(output_dir / "SN1987A_region_centres_to_query.csv")

plt.figure(figsize=(7, 6))

# Backprojected points
back = region_centres[region_centres["name"].str.contains("backprojected")]
plt.scatter(back["ra"], back["dec"], s=35, label="Back-projected 87A origin tests")

for _, row in back.iterrows():
    plt.text(row["ra"], row["dec"], f'{row["age_myr"]:g} Myr', fontsize=8)

# Current SN1987A and R136
plt.scatter([RA_87A], [DEC_87A], marker="*", s=160, label="SN1987A current")
plt.scatter([r136.ra.deg], [r136.dec.deg], marker="x", s=100, label="R136")

plt.gca().invert_xaxis()
plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.title("SN1987A current position, R136, and residual-PM back-projections")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Query the origin-point regions

Run this only after you are happy with the origin-point table.
It applies the same Gaia query to each search centre.

In [ ]:
def query_gaia_region(name, ra, dec, radius_deg=0.10, g_limit=19.0, overwrite=False):
    if not HAS_GAIA:
        raise ImportError("astroquery.gaia is not available in this environment.")

    safe_name = str(name).replace(" ", "_").replace(".", "p")
    out_file = input_dir / f"{safe_name}_raw_r{str(radius_deg).replace('.', 'p')}.csv"

    if out_file.exists() and not overwrite:
        print("Using existing:", out_file)
        return pd.read_csv(out_file, low_memory=False)

    query = f'''
    SELECT
        source_id,
        ra, ra_error,
        dec, dec_error,
        parallax, parallax_error,
        pmra, pmra_error,
        pmdec, pmdec_error,
        pmra_pmdec_corr,
        phot_g_mean_mag,
        phot_bp_mean_mag,
        phot_rp_mean_mag,
        bp_rp,
        ruwe,
        visibility_periods_used,
        duplicated_source,
        ipd_frac_multi_peak,
        ipd_gof_harmonic_amplitude
    FROM gaiadr3.gaia_source AS dr3
    WHERE 1 = CONTAINS(
        POINT('ICRS', dr3.ra, dr3.dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius_deg})
    )
    AND dr3.phot_g_mean_mag < {g_limit}
    '''

    print("Querying", name)
    job = Gaia.launch_job_async(query)
    tab = job.get_results()
    df_region = tab.to_pandas()
    df_region.to_csv(out_file, index=False)
    print("Saved", len(df_region), "rows to", out_file)
    return df_region

def clean_gaia_simple(df_region):
    x = df_region.copy()
    x.columns = x.columns.str.strip()

    for col in ["pmra", "pmdec", "parallax", "parallax_error", "ruwe"]:
        x = x[x[col].notna()]

    x = x[x["ruwe"] < 1.4]
    x = x[x["visibility_periods_used"] >= 10]
    x = x[x["duplicated_source"] == False]
    x = x[x["ipd_frac_multi_peak"] <= 10]
    x = x[x["ipd_gof_harmonic_amplitude"] <= 0.15]

    # Foreground rejection only.
    lmc_parallax_mas = 1.0 / D_LMC_KPC
    x = x[x["parallax"] - 3*x["parallax_error"] < lmc_parallax_mas]

    return x

# Example run:
# all_regions = []
# centres = pd.read_csv(output_dir / "SN1987A_region_centres_to_query.csv")
# for _, row in centres.iterrows():
#     raw = query_gaia_region(row["name"], row["ra"], row["dec"], radius_deg=0.10, g_limit=19.0)
#     clean = clean_gaia_simple(raw)
#     clean["region"] = row["name"]
#     clean["region_ra"] = row["ra"]
#     clean["region_dec"] = row["dec"]
#     all_regions.append(clean)
#
# all_regions = pd.concat(all_regions, ignore_index=True)
# all_regions.to_csv(output_dir / "SN1987A_all_origin_regions_clean.csv", index=False)
# all_regions.groupby("region").size()

## 7. R136 traceback sanity check

Your `SN1987A_R136_geometry.ipynb` already gives:
- projected separation ≈ 289 pc
- travel time 12 Myr → ≈ 23.5 km/s
- travel time 3 Myr → ≈ 94 km/s
- using the adopted PMs, closest approach ≈ 6.68 Myr ago at ≈ 174 pc

This cell reproduces the robust closest-approach calculation.

In [ ]:
def tangent_offset_mas(origin, target):
    sep = origin.separation(target).to_value(u.mas)
    pa = origin.position_angle(target).to_value(u.rad)  # east of north
    x_east = sep * np.sin(pa)
    y_north = sep * np.cos(pa)
    return x_east, y_north

def closest_approach_between_points(origin, target, pmra_origin, pmdec_origin, pmra_target, pmdec_target):
    x, y = tangent_offset_mas(origin, target)

    mu_x = pmra_target - pmra_origin
    mu_y = pmdec_target - pmdec_origin
    mu2 = mu_x**2 + mu_y**2

    t_past_yr = (x*mu_x + y*mu_y) / mu2
    d_closest_mas = np.sqrt((x - mu_x*t_past_yr)**2 + (y - mu_y*t_past_yr)**2)

    v_rel_kms = masyr_to_kms(np.sqrt(mu2))
    d_closest_pc = d_closest_mas * D_LMC_KPC / 206265.0

    return t_past_yr/1e6, d_closest_pc, v_rel_kms

sep = r136.separation(sn87a)
sep_pc = sep.radian * D_LMC_KPC * 1000.0

print("Current projected R136-SN1987A separation [pc]:", sep_pc)

for t_myr in [1.0, 1.8, 2.5, 3.0, 5.0, 10.0, 12.0, 14.0]:
    v_kms = sep_pc / t_myr / 1.022712
    mu_masyr = v_kms / (4.74047 * D_LMC_KPC)
    print(f"{t_myr:4.1f} Myr: {v_kms:7.1f} km/s, {mu_masyr:6.3f} mas/yr")

t_myr, d_pc, v_kms = closest_approach_between_points(
    origin=r136,
    target=sn87a,
    pmra_origin=PMRA_R136,
    pmdec_origin=PMDEC_R136,
    pmra_target=PMRA_87A,
    pmdec_target=PMDEC_87A
)

print()
print("Closest approach time [Myr ago]:", t_myr)
print("Closest approach distance [pc]:", d_pc)
print("Relative transverse velocity [km/s]:", v_kms)

## 8. Rough 8-solar-mass lifetime sanity check

Use this only as a quick reasoning table. For the thesis, replace this with MIST/PARSEC isochrones at LMC metallicity.

Interpretation:
- An 8 Msun progenitor is a long-lifetime, borderline core-collapse case.
- A 10-14 Myr age is more naturally associated with a higher-mass progenitor, roughly around 15-20 Msun in this rough table.

In [ ]:
anchor_mass = np.array([8, 9, 12, 15, 18, 20, 25, 40], dtype=float)
anchor_lifetime_myr = np.array([40, 30, 18, 13, 10.5, 9, 7, 4.5], dtype=float)

def lifetime_myr_rough(mass_msun):
    return 10 ** np.interp(
        np.log10(mass_msun),
        np.log10(anchor_mass),
        np.log10(anchor_lifetime_myr)
    )

mass_grid = np.linspace(8, 40, 200)
life_grid = lifetime_myr_rough(mass_grid)

lifetime_table = pd.DataFrame({
    "mass_msun": [8, 9, 12, 15, 18, 20, 25, 40],
    "rough_lifetime_myr": lifetime_myr_rough(np.array([8, 9, 12, 15, 18, 20, 25, 40]))
})

display(lifetime_table)

plt.figure(figsize=(6, 5))
plt.plot(mass_grid, life_grid)
plt.scatter(lifetime_table["mass_msun"], lifetime_table["rough_lifetime_myr"])
plt.axhspan(10, 14, alpha=0.2, label="10-14 Myr local-population check")
plt.axvline(8, linestyle="--", linewidth=1, label="8 Msun")
plt.xlabel("Initial mass [Msun]")
plt.ylabel("Rough lifetime [Myr]")
plt.title("Approximate massive-star lifetime sanity check")
plt.legend()
plt.tight_layout()
plt.show()